In [4]:
import pandas as pd
import numpy as np

# ------------------------------------
# 1. 데이터 로드
# ------------------------------------
df = pd.read_csv(
    "/Users/mac/Desktop/project/company_data/third_week/data_csv/전국일반음식점.csv",
    encoding="CP949",
    low_memory=False
)

# ------------------------------------
# 2. 날짜 처리
# ------------------------------------
df["인허가일자"] = pd.to_datetime(df["인허가일자"], errors="coerce")
df["폐업일자"] = pd.to_datetime(df["폐업일자"], errors="coerce")

df = df[df["인허가일자"] >= "2019-01-01"]

# ------------------------------------
# 3. 폐업 여부 생성
# ------------------------------------
df["폐업여부"] = df["폐업일자"].notnull().astype(int)

# ------------------------------------
# 4. 영업기간 생성
# ------------------------------------
today = pd.Timestamp("2025-01-01")
df["영업종료일"] = df["폐업일자"].fillna(today)
df["영업기간"] = (df["영업종료일"] - df["인허가일자"]).dt.days.clip(lower=0)

# ------------------------------------
# 5. Net Growth 계산
# ------------------------------------
df["year"] = df["인허가일자"].dt.year

annual = (
    df.groupby(["위생업태명", "year"])
      .agg(
          신규=("인허가일자", "count"),
          폐업=("폐업여부", "sum")
      )
      .reset_index()
)
annual["net_growth"] = annual["신규"] - annual["폐업"]

df = df.merge(
    annual[["위생업태명", "year", "net_growth"]],
    on=["위생업태명", "year"],
    how="left"
)

# ------------------------------------
# 6. 면적 처리
# ------------------------------------
df["소재지면적"] = pd.to_numeric(df["소재지면적"], errors="coerce")
df["소재지면적"] = df["소재지면적"].fillna(df["소재지면적"].median())
df["log_면적"] = np.log1p(df["소재지면적"])

# ------------------------------------
# 7. 지역 더미 생성
# ------------------------------------
def extract_region(addr):
    try:
        parts = addr.split()
        if len(parts) >= 2:
            return parts[0] + " " + parts[1]
        return np.nan
    except:
        return np.nan

df["지역"] = df["소재지전체주소"].astype(str).apply(extract_region)

# ------------------------------------
# 8. 원핫 인코딩
# ------------------------------------
df = pd.get_dummies(df, columns=["지역", "위생업태명"], drop_first=True)

# ------------------------------------
# 🔍 Dataset Summary 계산
# ------------------------------------

# 1) 최종 표본 수
total_samples = len(df)

# 2) 폐업 비율
폐업_count = df["폐업여부"].sum()
생존_count = total_samples - 폐업_count
폐업비율 = 폐업_count / total_samples * 100

# 3) 주요 변수 통계
영업기간_mean = df["영업기간"].mean()
영업기간_median = df["영업기간"].median()

net_growth_mean = df["net_growth"].mean()
net_growth_median = df["net_growth"].median()

log_area_mean = df["log_면적"].mean()
log_area_median = df["log_면적"].median()

# 4) 지역∙업종 dummy 수
지역_dummy_count = len([c for c in df.columns if c.startswith("지역_")])
업태_dummy_count = len([c for c in df.columns if c.startswith("위생업태명_")])

# ------------------------------------
# 🔍 출력
# ------------------------------------
print("최종 표본 수 :", total_samples)
print("폐업 비율 :", f"{폐업비율:.2f}%  (폐업 {폐업_count}, 생존 {생존_count})\n")

print("영업기간 평균 :", 영업기간_mean)
print("영업기간 중앙값 :", 영업기간_median)

print("Net Growth 평균 :", net_growth_mean)
print("Net Growth 중앙값 :", net_growth_median)

print("log_면적 평균 :", log_area_mean)
print("log_면적 중앙값 :", log_area_median)

print("\n지역 더미 수:", 지역_dummy_count)
print("업종 더미 수:", 업태_dummy_count)

최종 표본 수 : 425084
폐업 비율 : 38.80%  (폐업 164948, 생존 260136)

영업기간 평균 : 756.3179489230364
영업기간 중앙값 : 643.0
Net Growth 평균 : 8597.930761666368
Net Growth 중앙값 : 9813.0
log_면적 평균 : 3.9827225297764035
log_면적 중앙값 : 4.007333185232471

지역 더미 수: 224
업종 더미 수: 24
